# Where system dynamics is going in Python

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waelrash1/loopbricks/blob/main/notebooks/18-frontier.ipynb)

Much of the frontier of system dynamics is about letting the computer do more of the routine work: estimating parameters from data, hunting for flaws in a model, and searching for good policies. This notebook does a small version of each with plain Python loops: it fits the quality of work in the [Rework Cycle](https://waelrash1.github.io/loopbricks/#/m/reworkCycle) to noisy data, runs an automatic extreme conditions check over several molecules (among them [Reducing Backlog](https://waelrash1.github.io/loopbricks/#/m/reducingBacklog) and [Stock Management](https://waelrash1.github.io/loopbricks/#/m/stockmgmt)), and searches for the best ordering policy in Stock Management.

It goes with the [Where system dynamics is going module](https://waelrash1.github.io/loopbricks/#/learn/frontier) in the app.

In [ ]:
# Run this cell first. In Colab it installs the loopbricks package, which takes about ten seconds.
try:
    import loopbricks as lb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "loopbricks @ git+https://github.com/waelrash1/loopbricks.git@main#subdirectory=python"])
    import loopbricks as lb
import matplotlib.pyplot as plt
print("loopbricks", lb.__version__, "with molecules:", ", ".join(lb.molecules()))

## 1. Fitting a parameter to data

Projects run late in a predictable way. In the **Rework Cycle**, a team works through 1,000 tasks, but only a fraction (the `Quality`) is done right. The rest becomes undiscovered rework, which looks finished until it is found, some weeks later, and goes back on the pile. First see what quality does.

In [ ]:
import numpy as np
rw = lb.molecule("reworkCycle")
fig, ax = plt.subplots(figsize=(8, 4))
for q in (0.95, 0.8, 0.6):
    run = rw.run(T=40, quality=q)
    run["Work done"].plot(ax=ax, label=f"quality {q}")
    print(f"quality {q}: at week 16 work to do {run.loc[16.0, 'Work to do']:5.1f}, "
          f"undiscovered rework {run.loc[16.0, 'Undiscovered rework']:5.1f}, work done {run.loc[16.0, 'Work done']:5.1f}")
ax.set_ylabel("Work done"); ax.legend(); ax.grid(alpha=0.3)

At week 16 the visible pile of work to do is small in every case (under 20 tasks). What differs is the hidden stock: with 60% quality, about 153 tasks of undiscovered rework are waiting to surface, and work done is only 830. A manager watching the visible pile would think the project was nearly finished. The lateness comes from the structure, not from bad luck.

In a real project you do not know the quality; you have a record of work done, with errors in it. **Calibration** finds the parameter value that makes the model fit that record best. Here we make a fake record from a run with quality 0.7, add random errors, and then pretend we do not know the 0.7. The search is the simplest one there is: try every value on a grid and keep the one with the smallest sum of squared errors.

In [ ]:
rng = np.random.default_rng(7)
weeks = np.arange(0, 41, 2.0)                      # a report every two weeks
truth = rw.run(T=40, quality=0.7).loc[weeks, "Work done"].values
observed = truth + rng.normal(0, 25, len(weeks))   # reporting errors of about 25 tasks

def fit_quality(data):
    grid = np.round(np.arange(0.5, 0.96, 0.01), 2)
    errors = [((rw.run(T=40, quality=q).loc[weeks, "Work done"].values - data) ** 2).sum() for q in grid]
    return grid[int(np.argmin(errors))], grid, errors

best, grid, errors = fit_quality(observed)
print("best fitting quality:", best)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(grid, errors); ax1.set_xlabel("quality"); ax1.set_ylabel("sum of squared errors"); ax1.grid(alpha=0.3)
ax2.plot(weeks, observed, "o", label="noisy record")
rw.run(T=40, quality=best)["Work done"].plot(ax=ax2, label=f"model, quality {best}")
ax2.legend(); ax2.grid(alpha=0.3)

The search lands on 0.69, within 0.01 of the true value, and the error curve has a single clear valley. With one parameter and a fast model, trying every value is perfectly good; with dozens of parameters, the same idea needs smarter search methods, and that is where much current work (including machine learning) comes in.

**Try it.** Make the reporting errors four times larger (a standard deviation of 100 tasks). Fit ten different noisy records (seeds 0 to 9) and look at the spread of the estimates. How sure can you be of the quality now?

<details><summary>Show a worked answer</summary>

The next cell does it. The estimates range from 0.65 to 0.78, with most between 0.67 and 0.72. They still centre near the true 0.7, but any single fit could be off by as much as 0.08, which in this model is the difference between a project that is slightly late and one with a long tail of rework. A fit always comes with uncertainty, and fitting many noisy records (or bootstrapping one) is a simple way to see how much.
</details>

In [ ]:
estimates = []
for seed in range(10):
    noisy = truth + np.random.default_rng(seed).normal(0, 100, len(weeks))
    estimates.append(float(fit_quality(noisy)[0]))
print(estimates)

## 2. Software that tests the model for you

Extreme conditions tests are tedious by hand, which is exactly why software should run them. Here is a small automatic checker. For each constant in a model, it runs the model with that constant at a tenth and at ten times its usual value, and looks for one specific impossibility: an outflow that **asks a stock for more than it holds**. Stocks that cannot go negative are held at zero by the simulator, and every run records in `df.attrs["clipped"]` which stocks the floor had to catch and from when, so the checker only has to read that. (`warn_clips=False` stops each run printing its own warning.) The helper `changed` makes a copy of the model with one constant multiplied by a factor from a given week on; for now the change applies from the start of the run.

In [ ]:
def changed(model, name, factor, week=0):
    """A copy of the model in which the constant `name` is multiplied by `factor` from `week` on."""
    return model.with_equation(name, f"{model.params[name]:g} * (1 + STEP({factor - 1:g}, {week}))")

def auto_test(model, label, week=0):
    """Change every constant to a tenth and to ten times its value, from `week` on, and look for trouble."""
    problems = []
    for name in model.params:
        for factor in (0.1, 10):
            run = changed(model, name, factor, week).run(warn_clips=False)
            for stock, t in run.attrs["clipped"].items():     # stocks the floor at zero had to hold up
                problems.append(f"{name} x {factor}: {stock} held at zero from t = {t}")
    print(f"{label}: {'passes' if not problems else 'FAILS'}")
    for p in problems:
        print("   ", p)

for key in ("reducingBacklog", "doingWork", "protFlow", "backlogLevel", "stockmgmt"):
    auto_test(lb.molecule(key), key)

In a second or so the checker runs every constant at both extremes in five molecules. It flags **Reducing Backlog**, whose work keeps being "done" after the backlog is empty when the team or its productivity is ten times larger. That molecule is built to show exactly this flaw: its `Overdrawn` stock counts the work taken from nothing. **Doing Work**, **Level Protected by Flow** and **Stock Management** pass, because each limits its outflow by what the stock holds. In Stock Management, for example, sales are `MIN(Demand, Inventory / Minimum_shipping_time)`: in a week you cannot sell more than is on the shelf.

The flag on **Backlog Protected by Level** is a different kind of failure. Its shipping is limited by inventory too, but with ten times the orders the backlog keeps growing, and shipping would empty the inventory in `Full_service_inventory × Desired_ship_time / Backlog` months. Once the backlog passes about 2,400 (around month 23) that is under 0.05 months, half the time step of 0.1, so the Euler steps overshoot and swing, as in the [mechanics notebook](17-mechanics.ipynb), until the floor has to catch the stock at month 33. Run it with `dt=0.01` and the flag disappears: the checker found a time step problem, not a missing control. A flag is a reason to look, not a verdict.

A pass is weaker evidence than it looks, though. The checker tests one kind of failure, one constant at a time, in one kind of experiment.

**Try it.** Write a draft of Stock Management that forgets the limit, with `Sales = Demand`, and run the checker on it. Does it catch the flaw? If not, change the experiment so that each constant jumps to its extreme value at week 10 instead of from the start (`week=10`).

<details><summary>Show a worked answer</summary>

The next cell does both. Changed from the start, the draft **passes**: the model starts with its supply line full for whatever demand it is given, so even ten times the demand arrives with ten times the goods on order, and inventory never runs out. Changed at week 10, the draft **fails**: demand jumps from 10 to 100 a week, the goods on order are still sized for 10, and inventory is empty from week 11.3 to week 15.5 while the draft goes on "selling" 100 a week from an empty warehouse. (The run of the draft also prints the package's own warning that the floor held up `Inventory` from week 11.3.) The real molecule passes the same test: it never sells more in a week than the shelf holds, so sales drop to what is on hand (84 at week 10, about 48 by week 11.3, rising as deliveries catch up), and inventory never reaches zero.

So the flaw was there all along, but the first experiment could not reach it. Automatic checkers are only as good as the experiments they run, and tools that choose those experiments well, and link what they find back to the structure, are one of the open directions the module describes.
</details>

In [ ]:
import pandas as pd
sm = lb.molecule("stockmgmt")
draft = sm.with_equation("Sales", "Demand")   # the draft forgets that you can only sell what is in stock

auto_test(draft, "draft, changes from the start")
auto_test(draft, "draft, changes at week 10", week=10)
auto_test(sm, "stockmgmt, changes at week 10", week=10)

jump_draft = changed(draft, "demand", 10, week=10).run()
jump_real = changed(sm, "demand", 10, week=10).run()
print(pd.DataFrame({"draft inventory": jump_draft["Inventory"], "draft sales": jump_draft["Sales"],
                    "real inventory": jump_real["Inventory"], "real sales": jump_real["Sales"]})
      .loc[[9.9, 10, 11, 11.3, 13, 15.5, 20, 60]].round(1))

## 3. Searching for a better policy

Once a model is trusted for a purpose, the computer can search through policies for you. Take Stock Management with a 12-week lead time (the *Long lead time* preset). Inventory starts at 60, well below its target of 100, while the supply line starts full for current demand (12 weeks of 10 a week, so 120). The manager controls two things: the **supply line weight** (how much of the gap in goods on order to count) and the **inventory adjustment time** (how quickly to correct the inventory gap). We score each policy by the average distance of inventory from its target of 100 over 100 weeks: lower is better.

In [ ]:
import pandas as pd
sm = lb.molecule("stockmgmt")

def score(weight, adj_time):
    run = sm.run(T=100, preset="Long lead time", supply_line_weight=weight, inventory_adjustment_time=adj_time)
    return (run["Inventory"] - 100).abs().mean(), run["Orders"].max()

weights, times = (0, 0.25, 0.5, 0.75, 1), (2, 4, 8, 16)
results = {(w, t): score(w, t) for w in weights for t in times}
table = pd.DataFrame({f"weight {w}": [results[w, t][0] for t in times] for w in weights},
                     index=[f"adjust in {t} weeks" for t in times])
print("Average distance from target:")
print(table.round(2))
best = min(results, key=lambda k: results[k][0])
print(f"\nbest: weight {best[0]}, adjustment time {best[1]} (score {results[best][0]:.2f}, orders peak at {results[best][1]:.1f} a week)")

The best policy in the grid corrects inventory fast (2 weeks) and counts half of the supply line gap, with an average miss of 3.31 units; it overshoots the target only slightly (to about 100.5) and stays within 1 unit of it after week 18. Counting none of the supply line is worst in every row. A weight of 1 is not best here either: it brings inventory up without any overshoot, but slowly, and is not within 1 unit of the target until week 46. Counting only part of the supply line makes the manager order a little more, which closes the inventory gap sooner. The price is a little overshoot, and with a weight of 0 the overshoot grows into the cycle studied in [Oscillation](https://waelrash1.github.io/loopbricks/#/learn/oscillation).

A score also hides costs it does not measure. The best policy opens by ordering 30 a week, three times normal demand.

**Try it.** Suppose the supplier cannot take more than 20 orders a week. Search again, keeping only policies whose orders never go above 20. Which is best now, and how much worse is it?

<details><summary>Show a worked answer</summary>

The cell below filters the grid. Every policy with a 2-week adjustment time opens with orders of 30, so none is allowed. The best allowed one keeps the weight of 0.5 but adjusts in 4 weeks: an average miss of 4.49 instead of 3.31, with orders peaking at exactly 20. The answer to "what is the best policy" depends on what you count as a cost and what limits you respect, which is why policy search needs people to decide the goals, even when a computer does the searching.
</details>

In [ ]:
allowed = {k: v for k, v in results.items() if v[1] <= 20}
best = min(allowed, key=lambda k: allowed[k][0])
print(f"best with orders at most 20: weight {best[0]}, adjustment time {best[1]}, "
      f"score {allowed[best][0]:.2f}, orders peak at {allowed[best][1]:.1f}")

## Where next

- Back in the app: the [Where system dynamics is going module](https://waelrash1.github.io/loopbricks/#/learn/frontier) discusses open questions in theory, tools, implementation and education. The [Machine learning and AI with system dynamics](https://waelrash1.github.io/loopbricks/#/methods/ml-ai) page goes further than the grid searches here.
- The tests automated in section 2 are introduced one by one in [Testing and building confidence](https://waelrash1.github.io/loopbricks/#/learn/testing); fitting and sensitivity have their own pages, the [Calibration lab](https://waelrash1.github.io/loopbricks/#/methods/calibration) and [Sensitivity lab](https://waelrash1.github.io/loopbricks/#/methods/sensitivity).
- The rework cycle also appears in [The modelling process](https://waelrash1.github.io/loopbricks/#/learn/process), and the ordering rule in [Oscillation](https://waelrash1.github.io/loopbricks/#/learn/oscillation).